# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant


## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic metadata information
md = dataset.metadata
print(f"Dataset name: {md.name}\nDescription: {md.description}\nVersion: {md.version}\nLicense: {md.license}")
print(f"Published: {md.datePublished}")
print("Available record sets (by @id):", [r['@id'] for r in getattr(md, 'recordSet', [])])


## 2. Data Overview
Review available record sets, their fields, and IDs.

We will inspect the Croissant metadata structure to locate available record sets and field `@id`s. If the data package contains multiple record sets, we list their `@id`s for downstream operations.

In [ ]:
# List record sets and field IDs
record_sets = getattr(md, 'recordSet', [])
if not record_sets:
    print("No record sets found in the metadata. Dataset may not conform to Croissant full data structure or only contains a single default record set.")
else:
    for rs in record_sets:
        print(f"--- Record Set @id: {rs['@id']}")
        if 'field' in rs:
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            for f in fields:
                print(f"     Field @id: {f['@id']}, Name: {f.get('name')}, Data type: {f.get('dataType')}")
        else:
            print("     No fields found in this record set.")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

If the dataset does not specify record sets in its metadata, we attempt to infer or use the default provided by the library.

In [ ]:
# -- Identify record sets --
record_set_ids = [rs['@id'] for rs in getattr(md, 'recordSet', [])]

if not record_set_ids:
    # Fallback: If no recordSet present, use the main dataset @id
    record_set_ids = [md.__dict__.get('@id', 'main')] 
print("Using record set IDs:", record_set_ids)

# Extract data from each record set
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records from record set {rs_id}")
        dataframes[rs_id] = df
    except Exception as e:
        print(f"Failed to load records for record set {rs_id}: {e}")

# Show the first record set's columns and first few rows
default_rs_id = record_set_ids[0]
print(f"Columns in record set {default_rs_id}:\n", dataframes[default_rs_id].columns.tolist())
dataframes[default_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will select a numeric field (e.g., 'coverage_percent' or 'MW') present in the dataset to demonstrate basic filtering and normalization processes.

In [ ]:
# -- Identify a numeric field to analyze --

df = dataframes[default_rs_id]
numeric_candidates = [col for col in df.columns if df[col].dtype in [int, float, 'int64', 'float64'] or pd.api.types.is_numeric_dtype(df[col])]
if not numeric_candidates:
    # Try to guess by common names
    for name in [
        'coverage_percent', 'MW', 'molecular_weight', 'abundance', 'peptide_count', 'Peptides', 'Coverage (%)', 'MW (kDa)']:
        if name in df.columns:
            numeric_candidates = [name]
            break
if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Selected numeric field: {numeric_field}")
else:
    raise ValueError("No numeric field found for EDA.")

# Filter for values above an example threshold
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} records")

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by another non-numeric field, if available
group_field = None
for candidate in ['description', 'accession', 'Protein description', 'Sample', 'Type']:
    if candidate in df.columns:
        group_field = candidate
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nGrouped mean of {numeric_field} by {group_field} (showing first 5 groups):")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We will plot a histogram of the selected numeric field and, if available, a boxplot grouped by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f'Histogram of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

if group_field:
    plt.figure(figsize=(12,6))
    # Show only top 10 groups to avoid clutter
    top_groups = df[group_field].value_counts().index[:10]
    sns.boxplot(data=df[df[group_field].isin(top_groups)], x=group_field, y=numeric_field)
    plt.title(f'{numeric_field} distribution by {group_field} (top 10 groups)')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded and displayed dataset metadata using the Croissant schema and `mlcroissant`.
- Explored available record sets and their field structure using `@id` references.
- Loaded records into DataFrames referencing record set `@id`s.
- Identified and analyzed a numeric field, performing basic filtering, normalization, and grouping operations.
- Visualized the field distribution and conducted group-by EDA steps.

This workflow demonstrates how to dynamically explore and analyze FAIR datasets using Croissant schemas and the `mlcroissant` Python toolkit.